# Fine-Tuning Qwen3-TTS

Runs SFT training on the prepared JSONL, tests the checkpoint,
and synthesises the full story script with the fine-tuned voice.

**Prerequisite:** run `data_prep.ipynb` first — `finetune/train_with_codes.jsonl` must exist.

## 1. Setup

In [ ]:
import os, json, warnings, subprocess, sys
import torch, soundfile as sf
import IPython.display as ipd
from pathlib import Path
from pydub import AudioSegment
from safetensors.torch import load_file, save_file
from qwen_tts import Qwen3TTSModel

warnings.filterwarnings("ignore")  # silence noisy library warnings (torch/transformers)

# ── config ────────────────────────────────────────────────────────────────
REF_AUDIO    = Path("audio/ref_en.wav")           # source reference voice (will be resampled to 24k below)
OUTPUT_DIR   = Path("output")                     # generated wavs land here
CKPT_DIR     = Path("finetune/checkpoints")       # training writes per-epoch checkpoints here
OUT_CODES    = Path("finetune/train_with_codes.jsonl")  # training data produced by 02_data_prep
SCRIPT_PATH  = Path("script/llm_story.txt")       # full script to synthesize in step 4
SPEAKER_NAME = "laxmikant"                        # speaker id the checkpoint is trained/conditioned on
# ──────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Prepare & Train

In [5]:
ref_24k = REF_AUDIO.parent / (REF_AUDIO.stem + "_24k.wav")
if not ref_24k.exists():
    seg = AudioSegment.from_wav(str(REF_AUDIO))
    seg = seg.set_frame_rate(24000)  # training script expects a 24kHz reference
    seg = seg.set_channels(1)        # and mono
    seg.export(str(ref_24k), format="wav")

lines   = OUT_CODES.read_text(encoding="utf-8").splitlines()
patched = []
for line in lines:
    entry = json.loads(line)
    entry["ref_audio"] = str(ref_24k.resolve())  # repoint every entry at the resampled reference
    patched.append(json.dumps(entry, ensure_ascii=False))

OUT_CODES.write_text("\n".join(patched) + "\n", encoding="utf-8")  # overwrite in place with patched entries

571311

In [6]:
env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # reduce CUDA OOM from memory fragmentation

result = subprocess.run(
    [
        sys.executable, "finetune/scripts/finetuning/sft_12hz.py",
        "--init_model_path",   "Qwen/Qwen3-TTS-12Hz-1.7B-Base",  # base checkpoint to fine-tune from
        "--output_model_path", str(CKPT_DIR),                    # per-epoch checkpoints written here
        "--train_jsonl",       str(OUT_CODES),                   # patched training data from the previous cell
        "--batch_size",        "4",
        "--lr",                "2e-6",
        "--num_epochs",        "5",
        "--speaker_name",      SPEAKER_NAME,
    ],
    capture_output=True, text=True, env=env,
)

print(result.stdout[-3000:] if result.stdout else "")  # show the tail of the training log
if result.returncode != 0:
    print(result.stderr[-2000:])  # and the tail of the error if it failed


********
********
 
Epoch 0 | Step 0 | Loss: 13.4678
Epoch 0 | Step 10 | Loss: 13.2250
Epoch 0 | Step 20 | Loss: 12.2389
Epoch 1 | Step 0 | Loss: 11.3127
Epoch 1 | Step 10 | Loss: 11.2582
Epoch 1 | Step 20 | Loss: 10.2435
Epoch 2 | Step 0 | Loss: 9.7914
Epoch 2 | Step 10 | Loss: 10.4823
Epoch 2 | Step 20 | Loss: 9.3536
Epoch 3 | Step 0 | Loss: 9.7190
Epoch 3 | Step 10 | Loss: 7.7089
Epoch 3 | Step 20 | Loss: 9.9435
Epoch 4 | Step 0 | Loss: 9.2967
Epoch 4 | Step 10 | Loss: 9.4982
Epoch 4 | Step 20 | Loss: 9.2216



## 3. Test the Fine-Tuned Model

In [8]:
CHECKPOINT = "checkpoint-epoch-4"   # ← change to whichever checkpoint to test

ft_model = Qwen3TTSModel.from_pretrained(str((CKPT_DIR / CHECKPOINT).resolve()), device_map="cuda:0", dtype=torch.bfloat16)

wavs, sr = ft_model.generate_custom_voice(
    text="The custom voice or audio llm fine tuned system is working correctly. This is a quick sanity check.",
    language="English",
    speaker=SPEAKER_NAME,
    instruct="Speak naturally and clearly at a moderate pace.",
)

sf.write(str(OUTPUT_DIR / "finetuned_test.wav"), wavs[0], sr)  # save so you can compare checkpoints later
ipd.display(ipd.Audio(wavs[0], rate=sr))

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


## 4. Synthesize Full Script

In [9]:
# ── knobs ──────────────────────────────────────────────────────────────────
CHECKPOINT  = "checkpoint-epoch-2"   # ← change to whichever checkpoint to use
TEMPERATURE = 0.72                   # lower = more consistent voice
# ──────────────────────────────────────────────────────────────────────────

synth_model = Qwen3TTSModel.from_pretrained(str((CKPT_DIR / CHECKPOINT).resolve()), device_map="cuda:0", dtype=torch.bfloat16)

text = SCRIPT_PATH.read_text(encoding="utf-8").strip()  # full story script to narrate

wavs, sr = synth_model.generate_custom_voice(
    text=text,
    language="English",
    speaker=SPEAKER_NAME,
    instruct="Warm, engaging storytelling tone. Natural pace with gentle variation in pitch.",
    temperature=TEMPERATURE,
    top_p=0.88,             # nucleus sampling cutoff
    top_k=50,               # restrict sampling to the top-k candidate tokens
    repetition_penalty=1.1, # discourage repeated phrases over a long script
    max_new_tokens=4096,    # ceiling for a script this long
)

out_path = OUTPUT_DIR / f"{SCRIPT_PATH.stem}_finetuned.wav"
sf.write(str(out_path), wavs[0], sr)
ipd.display(ipd.Audio(wavs[0], rate=sr))

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


## 5. Push to Hugging Face

In [15]:
from huggingface_hub import HfApi, notebook_login
notebook_login()

In [11]:
CHECKPOINT = "checkpoint-epoch-4"

api = HfApi()
repo_id = "kgptalkie/qwen3-tts-finetuned-baraq-obama"

api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

api.upload_folder(
    folder_path=str((CKPT_DIR / CHECKPOINT).resolve()),
    repo_id=repo_id,
    repo_type="model",
)

model.safetensors:   0%|          | 0.00/682M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.83G [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/kgptalkie/qwen3-tts-finetuned-baraq-obama/commit/d995be92f924b125cc2933b9bd2ed1be3417f089', commit_message='Upload folder using huggingface_hub', commit_description='', oid='d995be92f924b125cc2933b9bd2ed1be3417f089', pr_url=None, repo_url=RepoUrl('https://huggingface.co/kgptalkie/qwen3-tts-finetuned-baraq-obama', endpoint='https://huggingface.co', repo_type='model', repo_id='kgptalkie/qwen3-tts-finetuned-baraq-obama'), pr_revision=None, pr_num=None)

In [13]:
from qwen_tts import Qwen3TTSModel
import torch

model = Qwen3TTSModel.from_pretrained("kgptalkie/qwen3-tts-finetuned-baraq-obama", device_map="cuda:0", dtype=torch.bfloat16)

wavs, sr = model.generate_custom_voice(
    text="There was once a little girl named Mia who asked too many questions.",
    language="English",
    speaker="laxmikant",
)


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


In [14]:
ipd.display(ipd.Audio(wavs[0], rate=sr))